# 01 · Recolección, limpieza y EDA

## Configuración inicial

| Dato | Valor |
|---|---|
| **Fecha de descarga** | 2026-10-26, 16:23 (hora de Lima) |
| **Medio elegido** | **B: El Comercio** |
| **URL base** | https://elcomercio.pe/ |

**URLs base por sección** (de cada una se toman las 90 noticias más recientes):

| Sección | URL de la sección | Fuente del listado (sitemap de noticias) | `etiqueta` |
|---|---|---|---|
| Economía | https://elcomercio.pe/economia/ | https://elcomercio.pe/sitemap/news/economia/?outputType=xml | `economia` |
| Política | https://elcomercio.pe/politica/ | https://elcomercio.pe/sitemap/news/politica/?outputType=xml | `politica` |
| DT (deporte) | https://elcomercio.pe/deporte-total/ | https://elcomercio.pe/sitemap/news/deporte-total/?outputType=xml | `deportes` |
| Ciencia y Tecnología | https://elcomercio.pe/tecnologia/ | https://elcomercio.pe/sitemap/news/tecnologia/?outputType=xml | `ciencia_tecnologia` |

## Objetivo

Construir `datos/noticias.csv` con las **360 noticias más recientes** (90 por sección). Si una sección no tiene suficientes noticias, el faltante se compensa con las secciones que tengan más publicadas.

## 0. Configuración (ejecutar primero)

Usa el kernel **Python (tarea_1)** (o `tarea_1/.venv/bin/python`). Ejecuta esta celda antes que cualquier otra: aquí se importan `pandas`, `requests`, `BeautifulSoup`, etc. Si reinicias el kernel, vuelve a ejecutarla.

In [1]:
import time
from datetime import datetime
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup
from tqdm.auto import tqdm
import plotly.express as px

DATOS = Path("datos")
RAW = DATOS / "raw"  # respaldo crudo de todo lo descargado
(RAW / "sitemaps").mkdir(parents=True, exist_ok=True)
(RAW / "html").mkdir(parents=True, exist_ok=True)

BASE = "https://elcomercio.pe"
MEDIO = "El Comercio"
TOTAL = 360
PAUSA = 0.5  # segundos de espera entre peticiones
SECCIONES = {  # sección en la web -> etiqueta oficial
    "economia": "economia",
    "politica": "politica",
    "deporte-total": "deportes",
    "tecnologia": "ciencia_tecnologia",
}

sesion = requests.Session()
sesion.headers["User-Agent"] = (
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/130 Safari/537.36"
)

/Users/susanmontalvo/Documents/GitHub/Diplomado-Su-2026/tarea_1/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Scraping

Todas las peticiones se hacen con `requests`, con una pausa de `PAUSA` segundos entre cada una, y cada respuesta se guarda tal cual en `datos/raw/` (respaldo crudo):

| Archivo | Contenido |
|---|---|
| `datos/raw/sitemaps/<seccion>.xml` | Listado de noticias de cada sección |
| `datos/raw/html/<seccion>_<nnn>.html` | Página HTML de cada noticia |
| `datos/raw/noticias_crudo.csv` | Datos extraídos, antes de la limpieza |

### 1.1 Listado de noticias por sección

El Comercio publica un sitemap de Google News por sección (`/sitemap/news/<seccion>/`) con las 100 notas más recientes: URL, titular y fecha de publicación.

In [2]:
def descargar(url: str, destino: Path, intentos: int = 3) -> bytes:
    """GET con reintentos: guarda la respuesta cruda en `destino` y luego hace una pausa."""
    for intento in range(intentos):
        try:
            resp = sesion.get(url, timeout=20)
            resp.raise_for_status()
            break
        except requests.RequestException:
            if intento == intentos - 1:
                raise
            time.sleep(2 * (intento + 1))
    destino.write_bytes(resp.content)
    time.sleep(PAUSA)  # ser respetuosos con el servidor
    return resp.content


def listar_seccion(seccion: str) -> pd.DataFrame:
    url = f"{BASE}/sitemap/news/{seccion}/?outputType=xml"
    sitemap = BeautifulSoup(descargar(url, RAW / "sitemaps" / f"{seccion}.xml"), "xml")
    filas = []
    for nodo in sitemap.find_all("url"):
        news = nodo.find("news")
        filas.append({
            "titular": news.find("title").get_text(strip=True),
            "url": nodo.find("loc").get_text(strip=True),
            "publicado": news.find("publication_date").get_text(strip=True),
            "seccion_web": seccion,
        })
    return pd.DataFrame(filas)


print("Fecha de descarga:", datetime.now().strftime("%Y-%m-%d %H:%M"))
listado = (
    pd.concat([listar_seccion(s) for s in SECCIONES], ignore_index=True)
    .sort_values("publicado", ascending=False)
    .drop_duplicates(subset="url")
)
listado["seccion_web"].value_counts()

Fecha de descarga: 2026-10-26 16:23


seccion_web
deporte-total    100
economia         100
politica         100
tecnologia        99
Name: count, dtype: int64

### 1.2 Cuota por sección

90 por sección (`TOTAL / 4`). Si alguna tiene menos, el faltante se asigna una a una a la sección con más noticias sobrantes.

In [3]:
def repartir_cuotas(disponibles: dict[str, int], total: int) -> dict[str, int]:
    cuotas = {s: min(n, total // len(disponibles)) for s, n in disponibles.items()}
    faltan = total - sum(cuotas.values())
    while faltan > 0:
        sobrantes = {s: disponibles[s] - cuotas[s] for s in cuotas if disponibles[s] > cuotas[s]}
        if not sobrantes:
            break
        cuotas[max(sobrantes, key=sobrantes.get)] += 1
        faltan -= 1
    return cuotas


cuotas = repartir_cuotas(listado["seccion_web"].value_counts().to_dict(), TOTAL)
seleccion = pd.concat(
    [listado[listado["seccion_web"] == s].head(n) for s, n in cuotas.items()], ignore_index=True
)
cuotas, len(seleccion)

({'deporte-total': 90, 'economia': 90, 'politica': 90, 'tecnologia': 90}, 360)

### 1.3 Bajada y sección original de cada nota

Se descarga el HTML de cada noticia (con su respaldo en `datos/raw/html/`): la bajada es la `meta description` y la sección original es el `section-id` de la página (p. ej. `/tecnologia/ciencias` → `tecnologia-ciencias`).

In [4]:
def meta(soup: BeautifulSoup, **attrs) -> str | None:
    tag = soup.find("meta", attrs=attrs)
    return tag["content"].strip() if tag and tag.get("content") else None


def detalle_noticia(url: str, archivo: Path) -> dict:
    try:
        soup = BeautifulSoup(descargar(url, archivo), "lxml")
    except requests.RequestException:
        return {"bajada": None, "seccion_original": None}
    ruta = meta(soup, name="section-id") or "/".join(url.removeprefix(BASE).split("/")[:-2])
    return {
        "bajada": meta(soup, name="description") or meta(soup, property="og:description"),
        "seccion_original": ruta.strip("/").replace("/", "-"),
    }


numero = seleccion.groupby("seccion_web").cumcount() + 1
seleccion["archivo_raw"] = seleccion["seccion_web"] + "_" + numero.map("{:03d}".format) + ".html"

detalles = [
    detalle_noticia(url, RAW / "html" / archivo)
    for url, archivo in tqdm(zip(seleccion["url"], seleccion["archivo_raw"]), total=len(seleccion), desc="Noticias")
]

df = pd.concat([seleccion, pd.DataFrame(detalles)], axis=1)
df.to_csv(RAW / "noticias_crudo.csv", index=False)  # respaldo antes de limpiar
df.shape

Noticias:   0%|          | 0/360 [00:00<?, ?it/s]

Noticias:   0%|          | 1/360 [00:00<03:41,  1.62it/s]

Noticias:   1%|          | 2/360 [00:01<03:57,  1.50it/s]

Noticias:   1%|          | 3/360 [00:02<04:00,  1.48it/s]

Noticias:   1%|          | 4/360 [00:02<03:49,  1.55it/s]

...

Noticias:  99%|█████████▉| 358/360 [08:29<00:02,  1.44s/it]

Noticias: 100%|█████████▉| 359/360 [08:30<00:01,  1.40s/it]

Noticias: 100%|██████████| 360/360 [08:31<00:00,  1.34s/it]

Noticias: 100%|██████████| 360/360 [08:31<00:00,  1.42s/it]

(360, 7)

## 2. Limpieza

In [5]:
df["medio"] = MEDIO
df["fecha"] = df["publicado"].str[:10]  # AAAA-MM-DD (hora de Lima)
df["etiqueta"] = df["seccion_web"].map(SECCIONES)
for col in ["titular", "bajada"]:
    df[col] = df[col].str.replace(r"\s+", " ", regex=True).str.strip()

df = df[["titular", "bajada", "medio", "seccion_original", "url", "fecha", "etiqueta"]]

assert len(df) >= 350 and df["url"].is_unique
print(df.isna().sum())
df["etiqueta"].value_counts()

titular             0
bajada              0
medio               0
seccion_original    0
url                 0
fecha               0
etiqueta            0
dtype: int64


etiqueta
deportes              90
economia              90
politica              90
ciencia_tecnologia    90
Name: count, dtype: int64

## 3. Exportación

Se guarda `datos/noticias.csv` con las 7 columnas y se vuelve a leer el archivo para comprobar lo que quedó guardado. Va antes del EDA porque el análisis se hace sobre este archivo.

In [6]:
ruta = DATOS / "noticias.csv"
df.to_csv(ruta, index=False)

df = pd.read_csv(ruta)  # se lee de nuevo para verificar el archivo exportado
assert df.shape[1] == 7

print("df.shape:", df.shape)
df.head()

df.shape: (360, 7)


,titular,bajada,medio,seccion_original,url,fecha,etiqueta
0,"Balón de Oro 2026 EN VIVO, ceremonia en directo",Balón de Oro 2026 EN VIVO: ver transmisión del...,El Comercio,deporte-total-futbol-mundial,https://elcomercio.pe/deporte-total/futbol-mun...,2026-10-26,deportes
1,¿A qué hora se da el Balón de Oro 2026 y quién...,"Horarios del inicio del Balón de Oro 2026 hoy,...",El Comercio,deporte-total-futbol-mundial,https://elcomercio.pe/deporte-total/futbol-mun...,2026-10-26,deportes
2,¿En qué canal EN VIVO transmiten Ceremonia del...,Revisa qué canales TV transmiten la ceremonia ...,El Comercio,deporte-total-futbol-mundial,https://elcomercio.pe/deporte-total/futbol-mun...,2026-10-26,deportes
3,DAZN EN VIVO GRATIS - Entrega del Balón de Oro...,DAZN EN VIVO GRATIS: ver transmisión de la Cer...,El Comercio,deporte-total-futbol-mundial,https://elcomercio.pe/deporte-total/futbol-mun...,2026-10-26,deportes
4,¿Quién ganó el Balón de Oro 2026? Resultados o...,Conoce quién fue el ganador de Balón de Oro 20...,El Comercio,deporte-total-futbol-mundial,https://elcomercio.pe/deporte-total/futbol-mun...,2026-10-26,deportes


## 4. Análisis exploratorio (EDA)

Tres gráficos con Plotly Express. El título de cada uno expresa la conclusión y se calcula a partir de los datos, así que sigue siendo correcto si se vuelven a descargar las noticias.

Esta celda lee `datos/noticias.csv`, por lo que basta con haber ejecutado la sección 0.

In [7]:
noticias = pd.read_csv(DATOS / "noticias.csv", parse_dates=["fecha"])

NOMBRES = {
    "economia": "Economía",
    "politica": "Política",
    "deportes": "Deportes",
    "ciencia_tecnologia": "Ciencia y Tecnología",
}
ORDEN = list(NOMBRES.values())
COLORES = dict(zip(ORDEN, ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]))  # mismo color por categoría en todos los gráficos
ESTILO = dict(template="plotly_white", barcornerradius=4, title_x=0.01)

noticias["categoria"] = noticias["etiqueta"].map(NOMBRES)
noticias.head()

,titular,bajada,medio,seccion_original,url,fecha,etiqueta,categoria
0,"Balón de Oro 2026 EN VIVO, ceremonia en directo",Balón de Oro 2026 EN VIVO: ver transmisión del...,El Comercio,deporte-total-futbol-mundial,https://elcomercio.pe/deporte-total/futbol-mun...,2026-10-26,deportes,Deportes
1,¿A qué hora se da el Balón de Oro 2026 y quién...,"Horarios del inicio del Balón de Oro 2026 hoy,...",El Comercio,deporte-total-futbol-mundial,https://elcomercio.pe/deporte-total/futbol-mun...,2026-10-26,deportes,Deportes
2,¿En qué canal EN VIVO transmiten Ceremonia del...,Revisa qué canales TV transmiten la ceremonia ...,El Comercio,deporte-total-futbol-mundial,https://elcomercio.pe/deporte-total/futbol-mun...,2026-10-26,deportes,Deportes
3,DAZN EN VIVO GRATIS - Entrega del Balón de Oro...,DAZN EN VIVO GRATIS: ver transmisión de la Cer...,El Comercio,deporte-total-futbol-mundial,https://elcomercio.pe/deporte-total/futbol-mun...,2026-10-26,deportes,Deportes
4,¿Quién ganó el Balón de Oro 2026? Resultados o...,Conoce quién fue el ganador de Balón de Oro 20...,El Comercio,deporte-total-futbol-mundial,https://elcomercio.pe/deporte-total/futbol-mun...,2026-10-26,deportes,Deportes


### 4.1 Distribución de noticias por categoría

In [8]:
por_categoria = (
    noticias["categoria"].value_counts().reindex(ORDEN)
    .rename_axis("categoria").reset_index(name="noticias")
)
por_categoria["porcentaje"] = por_categoria["noticias"] / por_categoria["noticias"].sum()

if por_categoria["noticias"].nunique() == 1:
    titulo = (f"Dataset balanceado: las {len(por_categoria)} categorías tienen "
              f"{por_categoria['noticias'].iloc[0]} noticias cada una")
else:
    mayor = por_categoria.loc[por_categoria["noticias"].idxmax()]
    titulo = f"{mayor.categoria} concentra el {mayor.porcentaje:.0%} de las noticias"

fig = px.bar(
    por_categoria, x="noticias", y="categoria", color="categoria", orientation="h", text="noticias",
    color_discrete_map=COLORES, hover_data={"porcentaje": ":.0%", "categoria": False},
    labels={"noticias": "Número de noticias", "categoria": "", "porcentaje": "Porcentaje"},
    title=titulo,
)
fig.update_layout(**ESTILO, showlegend=False, title_subtitle_text="Noticias por categoría (etiqueta)")
fig.update_yaxes(categoryorder="array", categoryarray=ORDEN[::-1])
fig.show()

### 4.2 Distribución de noticias por rango de fechas

Todas las noticias son de un solo medio (El Comercio), así que se analiza la distribución por fecha de publicación.

In [9]:
por_dia = noticias.groupby(["fecha", "categoria"]).size().reset_index(name="noticias")
dias = noticias.groupby("categoria")["fecha"].agg(lambda f: (f.max() - f.min()).days + 1)
rapida, lenta = dias.idxmin(), dias.idxmax()

titulo = (f"Las noticias de {rapida} caben en {dias[rapida]} días; "
          f"las de {lenta} abarcan {dias[lenta]}")
periodo = f"{noticias['fecha'].min():%d/%m} al {noticias['fecha'].max():%d/%m/%Y}"

fig = px.bar(
    por_dia, x="fecha", y="noticias", color="categoria",
    category_orders={"categoria": ORDEN}, color_discrete_map=COLORES,
    labels={"fecha": "Fecha de publicación", "noticias": "Número de noticias", "categoria": ""},
    title=titulo,
)
fig.update_layout(
    **ESTILO, bargap=0.15,
    title_subtitle_text=f"Noticias publicadas por día y categoría, del {periodo}",
)
fig.update_traces(marker_line_color="white", marker_line_width=1.5)
fig.update_xaxes(dtick="D1", tickformat="%d/%m")
fig.show()

dias.rename("días cubiertos").to_frame()

,días cubiertos
categoria,
Ciencia y Tecnología,15
Deportes,3
Economía,7
Política,7


### 4.3 Longitud de los titulares

In [10]:
noticias["largo_titular"] = noticias["titular"].str.len()
q1, mediana, q3 = noticias["largo_titular"].quantile([0.25, 0.5, 0.75]).round().astype(int)

fig = px.histogram(
    noticias, x="largo_titular", nbins=25, color_discrete_sequence=[COLORES["Economía"]],
    labels={"largo_titular": "Longitud del titular (caracteres)"},
    title=f"La mitad de los titulares mide entre {q1} y {q3} caracteres (mediana: {mediana})",
)
fig.add_vline(x=mediana, line_dash="dash", line_color="#52514e",
              annotation_text=f"Mediana: {mediana}", annotation_position="top right")
fig.update_layout(
    **ESTILO, bargap=0.05, yaxis_title="Número de noticias",
    title_subtitle_text="Distribución de la longitud de los titulares",
)
fig.show()

noticias["largo_titular"].describe().round(1)

count    360.0
mean      95.7
std       25.2
min       13.0
25%       82.8
50%       97.0
75%      108.0
max      167.0
Name: largo_titular, dtype: float64

### 4.4 Validación: titulares más largos y más cortos

Se muestran los 5 titulares más largos y los 5 más cortos y se verifica que coincidan con el histograma: deben ser los extremos del eje X y, si son atípicos, quedar fuera del rango `Q1 − 1.5·IQR` a `Q3 + 1.5·IQR`.

In [11]:
COLUMNAS = ["largo_titular", "titular", "categoria", "seccion_original"]
mas_largos = noticias.nlargest(5, "largo_titular")[COLUMNAS]
mas_cortos = noticias.nsmallest(5, "largo_titular")[COLUMNAS]

with pd.option_context("display.max_colwidth", None):
    print("5 titulares más largos")
    display(mas_largos)
    print("5 titulares más cortos")
    display(mas_cortos)

5 titulares más largos


,largo_titular,titular,categoria,seccion_original
88,167,"“Tiene habilidad, falta musculatura” y “Aún es parte del universo de convocables”: cinco voces autorizadas sobre el futuro de Maxloren Castro en Cristal y la selección",Deportes,deporte-total-futbol-peruano
11,165,"“Para una hipotética final Alianza pediría árbitro y VAR extranjeros, además de goal line”: Alfredo Arosemena y su postura sobre el arbitraje, Paolo, Guede y Velazco",Deportes,deporte-total-futbol-peruano
26,160,El “Quedan 4 finales” de Guede y el cántico de 5 mil hinchas que coparon Moquegua: lo que no se vio del dramático 1-0 que mantiene vivo a Alianza en el Clausura,Deportes,deporte-total-futbol-peruano
74,158,“El culpable soy yo” de Mosquera y por qué sacó a Yotún con el 2-0 en contra: lo que no se vio del tropezón de Cristal en Cusco que lo deja en la cuerda floja,Deportes,deporte-total-futbol-peruano
241,157,Tomás Gálvez pide derogar ley que permite a la PNP investigación preliminar sin fiscales: “Ha generado un conflicto entre el Ministerio Público y la Policía”,Política,politica-justicia


5 titulares más cortos


,largo_titular,titular,categoria,seccion_original
208,13,Sillas vacías,Política,opinion-columnistas
258,15,El rol de Obras,Política,politica-opinion
206,17,El dictador feliz,Política,opinion-columnistas
97,25,El Niño no es la pandemia,Economía,economia-dia-1
172,27,Las muchas caras de El Niño,Economía,economia-opinion


In [12]:
iqr = q3 - q1
limite_inf, limite_sup = q1 - 1.5 * iqr, q3 + 1.5 * iqr
minimo, maximo = noticias["largo_titular"].min(), noticias["largo_titular"].max()

# Los extremos de las tablas deben ser los extremos del eje X del histograma
assert mas_cortos["largo_titular"].iloc[0] == minimo and mas_largos["largo_titular"].iloc[0] == maximo

print(f"Histograma: de {minimo} a {maximo} caracteres; mitad central entre {q1} y {q3} (mediana {mediana})")
print(f"Rango sin atípicos: {limite_inf:.0f} a {limite_sup:.0f} caracteres\n")
for nombre, grupo, limite, fuera in [
    ("Más cortos", mas_cortos, limite_inf, mas_cortos["largo_titular"] < limite_inf),
    ("Más largos", mas_largos, limite_sup, mas_largos["largo_titular"] > limite_sup),
]:
    # columna de opinión: lo indica la sección o la URL
    opinion = noticias.loc[grupo.index, ["seccion_original", "url"]].apply(lambda s: s.str.contains("opinion")).any(axis=1).sum()
    print(f"{nombre}: {grupo['largo_titular'].tolist()} caracteres -> "
          f"{fuera.sum()} de 5 son atípicos (límite {limite:.0f}); {opinion} de 5 son columnas de opinión")

Histograma: de 13 a 167 caracteres; mitad central entre 83 y 108 (mediana 97)
Rango sin atípicos: 46 a 146 caracteres

Más cortos: [13, 15, 17, 25, 27] caracteres -> 5 de 5 son atípicos (límite 46); 5 de 5 son columnas de opinión
Más largos: [167, 165, 160, 158, 157] caracteres -> 5 de 5 son atípicos (límite 146); 0 de 5 son columnas de opinión


**Conclusión de la validación** (descarga del 26/10/2026)

- Las tablas coinciden con el histograma: el titular más corto (13 caracteres) y el más largo (167) son los extremos del eje X, y los 10 están en las colas, fuera del rango sin atípicos (46 a 146).
- **Los más cortos son columnas de opinión** (*"Sillas vacías"*, *"El dictador feliz"*): usan títulos breves y metafóricos, a diferencia de las notas informativas. Explican la pequeña cola izquierda del histograma.
- **Los más largos son notas informativas, casi todas de Deportes** (fútbol peruano): combinan citas textuales con el contexto del partido, lo que alarga el titular.
- No hay titulares vacíos ni cortados: la longitud mínima corresponde a títulos reales.

## Conclusiones

Descarga del 26/10/2026, medio B (El Comercio).

1. **Extracción:** se obtuvieron **360 noticias** (mínimo exigido: 350) de 4 secciones, usando los sitemaps de noticias por sección para el listado y el HTML de cada nota para la bajada y la sección original. Hubo pausas entre todas las peticiones y quedó un respaldo crudo en `datos/raw/`.
2. **Calidad:** el dataset final tiene 7 columnas, sin valores vacíos ni URLs duplicadas, y se exportó a `datos/noticias.csv`.
3. **Balance:** las 4 categorías tienen 90 noticias cada una (25 %), así que no hace falta corregir desbalance de clases en el modelado.
4. **Ritmo de publicación:** con la misma cuota, Deportes cubre solo 3 días (unas 30 notas diarias), Economía y Política 7, y Ciencia y Tecnología 15. El periodo de cada categoría es distinto, lo que puede influir en los temas que aparecen (por ejemplo, el Balón de Oro en Deportes).
5. **Titulares:** la mitad mide entre 83 y 108 caracteres (mediana 97). La validación confirmó que los extremos del histograma son reales: los más cortos (13 a 27 caracteres) son columnas de opinión y los más largos (157 a 167) son notas de fútbol con citas textuales.
6. **Para la clasificación:** 15 de las 360 noticias son columnas de opinión publicadas en Política y Economía. Su estilo (títulos breves y metafóricos) puede hacerlas más difíciles de clasificar solo por el titular.